# V48 Colab版 — pseudo-label生成

3-fold YOLOアンサンブル(2/3合意フィルタ)+ all_data refiner + **確定後処理
shrink@0.45**(free variantは監査でfold0選択バイアスと判明、使わない)で
GONG追加画像150枚にpseudo-labelを付与する。

ランタイム: **GPU**(L4/A100等)。


In [ ]:
# [1/5] 環境確認・依存関係・Drive mount
import json, math, time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
from google.colab import drive
from pycocotools import mask as mask_utils

drive.mount("/content/drive")

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("gpu:", gpu_name)
if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを選択してください")

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
from ultralytics import YOLO

class CFG:
    V30_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    V35_ROOT = Path("/content/drive/MyDrive/solar_filament_v35")
    ARTIFACTS_ROOT = V30_ROOT / "artifacts"
    YOLO_WEIGHTS = {f: V30_ROOT / "runs" / f"yolo_fold_{f}" / "weights" / "best.pt" for f in (0, 1, 2)}
    REFINER_CKPT = ARTIFACTS_ROOT / "mediseg_refiner_all_data_resume.pt"
    PSEUDO_DIR = V35_ROOT / "pseudo_label" / "preprocessed"
    OUT_DIR = V35_ROOT / "pseudo_label" / "cache"
    CROP_SIZE = 320
    CONF_YOLO = 0.10          # WBF融合前は緩めに集める(融合後にconf_finalで絞る)
    MIN_AREA = 20
    CONF_FINAL = 0.50         # 本番0.26より厳しいpseudo-label採用閾値
    MIN_SOURCE_AGREEMENT = 2  # 3-foldのうち2つ以上が同一個体を検出した場合のみ採用
    WBF_IOU_THRESHOLD = 0.5
    WBF_VOTE_THRESHOLD = 0.5
    SHRINK_THRESHOLD = 0.45   # 確定構成(free variantではない)
    DROP_THRESHOLD = 0.90

for required in list(CFG.YOLO_WEIGHTS.values()) + [CFG.REFINER_CKPT, CFG.PSEUDO_DIR]:
    if not required.exists():
        raise FileNotFoundError(f"{required} が見つからない")

CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
targets = sorted(p.name for p in CFG.PSEUDO_DIR.glob("*.jpeg"))
print({"pseudo_images": len(targets), "gpu": gpu_name})

In [ ]:
# [2/5] MediSeg UNetアーキテクチャ・WBF(solar_filament/と同一)
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)


"""複数seed(独立に学習した検出器)のmask提案を統合するWeighted Mask Fusion。

## 動機

multi-seedアンサンブル(V42)は単純連結(全proposalをそのままconcat)で
TP+53よりFP+190の方が大きく、正味で悪化した(改訂9 §27.5)。それぞれの
seedに固有の誤検出が、確定後処理のdrop_contained(IoU≥0.90の重複除去のみ)
では吸収しきれなかったため。

WBF(Weighted Boxes Fusion、本来はbox用)のmask版。IoUでクラスタ化し、
クラスタ内のmaskを confidence 加重平均→0.5で二値化して1個の統合maskにする。
2つのseedが同じ個体を検出した場合はconfidence加重平均で滑らかに統合され、
片方のseedだけの誤検出は単独クラスタのまま残る(スコアで後段のconf閾値に
掛けられる)ので、単純連結よりFPを抑えられるという仮説を検証する。
"""


import numpy as np
from pycocotools import mask as mask_util


def _mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    intersection = np.logical_and(a, b).sum()
    if intersection == 0:
        return 0.0
    union = np.logical_or(a, b).sum()
    return float(intersection) / float(union) if union else 0.0


def weighted_mask_fusion(
    masks: list[np.ndarray],
    scores: np.ndarray,
    source_ids: list[int] | None = None,
    iou_threshold: float = 0.5,
    vote_threshold: float = 0.5,
) -> tuple[list[np.ndarray], np.ndarray, list[int]]:
    """複数seed分のmask提案をIoUクラスタ化し、加重平均で統合する。

    Args:
        masks: 統合対象のboolマスク(全seed分をあらかじめ結合したもの)。
        scores: 各マスクのconfidence。
        source_ids: 各マスクの由来seed(任意)。同一クラスタ内の由来seed数を
            数えるためだけに使う(合意度の記録用、フュージョン計算自体には未使用)。
        iou_threshold: この値以上のIoUを同一個体とみなしてクラスタ化する。
            drop_containedの0.90より緩い値を想定(異なるseedの検出は
            境界が微妙にずれるため、厳しすぎると同一個体でも別クラスタになる)。
        vote_threshold: confidence加重平均マスクをこの閾値で二値化する。
            0.5より低いと統合maskが大きくなる(overcoverage方向)、
            高いと小さくなる(undercoverage方向)。

    Returns:
        (統合後のmaskリスト, 統合後のscore配列, 各クラスタの由来seed数リスト)
    """
    if not masks:
        return [], np.zeros(0, np.float32), []
    if source_ids is None:
        source_ids = [0] * len(masks)

    order = list(np.argsort(-scores))
    clusters: list[dict] = []  # {"rep": mask, "members": [idx,...]}

    for index in order:
        mask = masks[index]
        best_cluster = None
        best_iou = iou_threshold
        for cluster in clusters:
            iou = _mask_iou(mask, cluster["rep"])
            if iou >= best_iou:
                best_cluster = cluster
                best_iou = iou
        if best_cluster is None:
            clusters.append({"rep": mask, "members": [index]})
        else:
            best_cluster["members"].append(index)
            # 代表maskは最高スコアのまま固定(orderがスコア降順なので先頭が最高)

    fused_masks: list[np.ndarray] = []
    fused_scores: list[float] = []
    fused_source_counts: list[int] = []
    for cluster in clusters:
        members = cluster["members"]
        member_scores = np.array([scores[i] for i in members], dtype=np.float64)
        weights = member_scores / member_scores.sum()
        accum = np.zeros(masks[members[0]].shape, dtype=np.float64)
        for member_index, weight in zip(members, weights):
            accum += masks[member_index].astype(np.float64) * weight
        fused_mask = accum >= vote_threshold
        if fused_mask.sum() == 0:
            continue
        fused_masks.append(fused_mask)
        fused_scores.append(float(member_scores.mean()))
        fused_source_counts.append(len({source_ids[i] for i in members}))

    return fused_masks, np.asarray(fused_scores, dtype=np.float32), fused_source_counts

def soft_dice_loss(logits, targets):
    probs = logits.sigmoid()
    intersection = (probs * targets).sum((1, 2, 3))
    denominator = probs.sum((1, 2, 3)) + targets.sum((1, 2, 3))
    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()

def imread_unicode(path):
    return cv2.imdecode(np.frombuffer(Path(path).read_bytes(), np.uint8), cv2.IMREAD_COLOR)


In [ ]:
# [3/5] YOLO 3-fold・refinerのロード
yolo_models = {f: YOLO(str(path)) for f, path in CFG.YOLO_WEIGHTS.items()}

refiner_ckpt = torch.load(CFG.REFINER_CKPT, map_location="cpu", weights_only=False)
refiner = MediSegUNet(3, 1, False).cuda().eval()
refiner.load_state_dict(refiner_ckpt.get("best_model") or refiner_ckpt["model"], strict=True)
print("loaded: 3x YOLO fold models + all_data refiner")

def square_box(mask, context=1.65, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

def refine_batch(refiner_model, image, proposals, batch=32):
    boxes, crops = [], []
    for proposal in proposals:
        x0, y0, x1, y1 = square_box(proposal)
        crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
        boxes.append((x0, y0, x1, y1))
        crops.append(crop.transpose(2, 0, 1))
    probabilities = []
    with torch.no_grad():
        for start in range(0, len(crops), batch):
            chunk = np.stack(crops[start:start + batch])
            tensor = torch.from_numpy(chunk.copy()).float().cuda() / 255.0
            output = refiner_model(tensor).sigmoid()[:, 0].cpu().numpy()
            for offset in range(output.shape[0]):
                x0, y0, x1, y1 = boxes[start + offset]
                probabilities.append(cv2.resize(output[offset], (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR))
    return boxes, probabilities

def shrink_variant(proposal, box, probability, threshold):
    x0, y0, x1, y1 = box
    refined = np.zeros_like(proposal, dtype=bool)
    refined[y0:y1, x0:x1] = probability >= threshold
    return refined & proposal  # shrink = refined AND proposal(確定構成)

def greedy_disjoint_local(masks, order, min_area=CFG.MIN_AREA, canvas=2048):
    occupied = np.zeros((canvas, canvas), dtype=bool)
    kept = []
    for index in order:
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < min_area:
            continue
        kept.append(exclusive)
        occupied |= exclusive
    return kept

def drop_contained_indices_local(masks, threshold):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            value = self_iou[i, j]
            if value <= 0:
                continue
            intersection = value * (areas[i] + areas[j]) / (1.0 + value)
            smaller = i if areas[i] <= areas[j] else j
            if intersection / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership_local(masks, threshold):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    subset = drop_contained_indices_local(masks, threshold)
    return greedy_disjoint_local(masks, sorted(subset, key=lambda i: areas[i]))


In [ ]:
# [4/5] 画像ごとに: 3-fold YOLO推論 -> WBF融合(2/3合意フィルタ) -> refiner(shrink@0.45) -> 排他化
started = time.perf_counter()
manifest = []
for position, name in enumerate(targets, 1):
    out_path = CFG.OUT_DIR / f"{Path(name).stem}.npz"
    image_path = CFG.PSEUDO_DIR / name
    image = imread_unicode(image_path)

    all_masks, all_scores, all_sources = [], [], []
    for fold, model in yolo_models.items():
        result = model.predict(str(image_path), conf=CFG.CONF_YOLO, verbose=False)[0]
        if result.masks is None:
            continue
        for mask_xy, box_conf in zip(result.masks.data.cpu().numpy(), result.boxes.conf.cpu().numpy()):
            mask_full = cv2.resize(mask_xy, (2048, 2048), interpolation=cv2.INTER_LINEAR) >= 0.5
            if mask_full.sum() < CFG.MIN_AREA:
                continue
            all_masks.append(mask_full)
            all_scores.append(float(box_conf))
            all_sources.append(fold)

    if not all_masks:
        np.savez_compressed(out_path, packed=np.empty((0, 0, 0), np.uint8),
                            shape=np.asarray([2048, 2048], np.int32), confidences=np.zeros(0, np.float32),
                            agreement=np.zeros(0, np.int32))
        manifest.append({"file_name": name, "n_proposals": 0, "n_kept": 0})
        continue

    fused_masks, fused_scores, fused_agreement = weighted_mask_fusion(
        all_masks, np.asarray(all_scores, dtype=np.float32), all_sources,
        iou_threshold=CFG.WBF_IOU_THRESHOLD, vote_threshold=CFG.WBF_VOTE_THRESHOLD)

    # 2/3以上のfoldが合意 かつ confidence >= 0.5 のみ採用
    keep_indices = [i for i in range(len(fused_masks))
                    if fused_agreement[i] >= CFG.MIN_SOURCE_AGREEMENT and fused_scores[i] >= CFG.CONF_FINAL]
    proposals = [fused_masks[i] for i in keep_indices]
    kept_agreement = [fused_agreement[i] for i in keep_indices]
    kept_scores = [float(fused_scores[i]) for i in keep_indices]

    if not proposals:
        np.savez_compressed(out_path, packed=np.empty((0, 0, 0), np.uint8),
                            shape=np.asarray([2048, 2048], np.int32), confidences=np.zeros(0, np.float32),
                            agreement=np.zeros(0, np.int32))
        manifest.append({"file_name": name, "n_proposals": len(fused_masks), "n_kept": 0})
        continue

    boxes, probabilities = refine_batch(refiner, image, proposals)
    refined = [shrink_variant(proposals[i], boxes[i], probabilities[i], CFG.SHRINK_THRESHOLD)
               for i in range(len(proposals))]
    final_order = sorted(range(len(refined)), key=lambda i: -kept_scores[i])
    final_masks_ordered = [refined[i] for i in final_order]
    final = apply_ownership_local(final_masks_ordered, CFG.DROP_THRESHOLD)
    # apply_ownership_localはarea_asc昇順に並べ替えて処理するため、final要素順はsize順。
    # confidence/agreementは元のmaskとサイズで対応付け直すのではなく、
    # final各要素についてどのproposalに由来するか面積一致で逆引きする簡易対応。
    final_scores, final_agreement = [], []
    for f_mask in final:
        best_i, best_overlap = 0, -1.0
        for i, r_mask in enumerate(final_masks_ordered):
            overlap = np.logical_and(f_mask, r_mask).sum()
            if overlap > best_overlap:
                best_overlap, best_i = overlap, i
        final_scores.append(kept_scores[final_order[best_i]])
        final_agreement.append(kept_agreement[final_order[best_i]])

    if final:
        packed = np.packbits(np.stack(final).astype(np.uint8), axis=-1)
        shape = np.asarray(final[0].shape, np.int32)
    else:
        packed = np.empty((0, 0, 0), np.uint8)
        shape = np.asarray([2048, 2048], np.int32)
    np.savez_compressed(out_path, packed=packed, shape=shape,
                        confidences=np.asarray(final_scores, np.float32),
                        agreement=np.asarray(final_agreement, np.int32))
    manifest.append({"file_name": name, "n_proposals": len(fused_masks), "n_kept": len(final)})

    if position % 25 == 0 or position == len(targets):
        total_kept = sum(m["n_kept"] for m in manifest)
        print(f"  {position}/{len(targets)}  {time.perf_counter()-started:.0f}s  累計採用{total_kept}件", flush=True)

manifest_path = CFG.OUT_DIR.parent / "pseudo_label_manifest.json"
manifest_path.write_text(json.dumps({
    "scope": f"{len(targets)}枚のGONG追加画像。3-fold YOLO WBF(2/3合意)+refiner(shrink@0.45確定構成)",
    "conf_final": CFG.CONF_FINAL, "min_source_agreement": CFG.MIN_SOURCE_AGREEMENT,
    "total_kept": sum(m["n_kept"] for m in manifest),
    "images": manifest,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved manifest:", manifest_path)
print("total_kept:", sum(m["n_kept"] for m in manifest), "/ ", len(targets), "images")

In [ ]:
# [5/5] QA用サマリ: 検出数分布・信頼度分布を確認(人間QAビューアへ渡す前の一次チェック)
kept_counts = [m["n_kept"] for m in manifest]
print({
    "images_with_zero_pseudo_labels": sum(1 for c in kept_counts if c == 0),
    "mean_kept_per_image": float(np.mean(kept_counts)),
    "max_kept_per_image": max(kept_counts),
    "total_pseudo_labels": sum(kept_counts),
})